In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd


# Paths
INPUT_PATH = Path("data/raw/combined_bitcoin_price_2020_2024.xlsx")
OUTPUT_PATH = Path("data/processed/bitcoin_with_volatility.csv")


# Load raw Bitcoin market data
df = pd.read_excel(INPUT_PATH)


# Convert OHLC columns to numeric values
for col in ["open", "high", "low", "close"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")


# Check missing and zero values in the OHLC columns
key_columns = ["open", "high", "low", "close"]

print("Checking NaN and zero values in OHLC data:\n")

for col in key_columns:
    num_missing = df[col].isna().sum()
    num_zero = (df[col] == 0).sum()
    print(f"{col}: NaN = {num_missing} | zero = {num_zero}")


# Rogers-Satchell volatility
rs_variance = (
    np.log(df["high"] / df["close"]) * np.log(df["high"] / df["open"])
    + np.log(df["low"] / df["close"]) * np.log(df["low"] / df["open"])
)

df["rs_volatility"] = np.sqrt(rs_variance)


# Check the resulting volatility series
print("\nChecking rs_volatility:\n")
print("NaN:", df["rs_volatility"].isna().sum())
print("Zero or negative:", (df["rs_volatility"] <= 0).sum())

print("\nSummary statistics for rs_volatility:\n")
print(df["rs_volatility"].describe())


# Save processed data
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUTPUT_PATH, index=False)

print(f"\nSaved processed data to: {OUTPUT_PATH}")